# 7.6 Get Your Survey Data Machine Learning Ready


## Introduction


In our previous modules, we focused on "reading" the student voice through Topic Modeling and Sentiment Analysis. Now, we must translate those insights into a format that can live alongside our structured institutional data.

This notebook serves as the strategic "bridge" between unstructured text mining and advanced machine learning. We aren't just cleaning data here; we are performing **Feature Fusion**: the process of combining student demographics, academic performance, and compressed linguistic patterns into a single, high-dimensional matrix.

**In this module, we will focus on:**
* **Multimodal Integration:** Preparing numeric, categorical, and text data to "speak the same language" using a unified preprocessing pipeline.
* **Dimensionality Reduction (PCA):** Solving the "Curse of Dimensionality" by compressing hundreds of TF-IDF word features into a concentrated set of Principal Components.
* **The Master Matrix:** Exporting a consolidated dataset that will power the predictive models and student segmentation analysis in the upcoming sections of the course.

<br>

> **Instructor Perspective:** In earlier courses, the focus was on the *mechanics* of scaling and encoding. Now, the challenge is *balance*. If we leave text data as raw TF-IDF vectors, the hundreds of word columns will mathematically "drown out" our vital GPA and demographic markers. **PCA acts as our equalizer**, ensuring every data source has a fair voice in the final model.



## 1. Setup and Data Preparation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import random
import numpy as np
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

pd.options.display.max_columns = None
np.random.seed(42)
random.seed(42)

filepath = '/content/drive/MyDrive/IR ML Cert/MLCert Course 3/Course 3 Data/'
df_training_assignment = pd.read_csv('/content/drive/MyDrive/Applied-Data-Analytics-For-Higher-Education-Course-3/data/training_assignment.csv')
df_testing_assignment = pd.read_csv('/content/drive/MyDrive/Applied-Data-Analytics-For-Higher-Education-Course-3/data/testing_assignment.csv')


print("Training size:", len(df_training_assignment))
print("Test size:", len(df_testing_assignment))


## 2. Preprocess Structured Student Data




As established in **Course 2**, machine learning algorithms perform best when numerical features are on a similar scale and categorical variables are transformed into binary indicators.

In this module, we use a `ColumnTransformer` to apply three distinct strategies based on the nature of our institutional data:

| Feature Type | Columns | Scaling Strategy | IR Rationale |
| :--- | :--- | :--- | :--- |
| **Academic Ratios** | HS_GPA, Term GPAs, DFW Rates | **MinMaxScaler** | Preserves the natural 0.0–4.0 or 0–1 boundaries. |
| **Volume Counts** | Units Attempted | **StandardScaler** | Centers data around the mean; useful for identifying credit-load outliers. |
| **Demographics** | Gender, Ethnicity, First-Gen | **OneHotEncoder** | Converts categories into "dummy variables" for mathematical processing. |

<br>

> **Instructor Perspective:** We aren't just cleaning data here; we are ensuring that a "4.0 GPA" and "15 Units Attempted" have equal weighting in the eyes of the model. Without this step, the larger numbers (units) would mathematically overwhelm the smaller numbers (GPA).


In [ ]:
minmax_cols = ['HS_GPA', 'GPA_1', 'GPA_2', 'DFW_RATE_1', 'DFW_RATE_2']
standard_cols = ['UNITS_ATTEMPTED_1', 'UNITS_ATTEMPTED_2']
categorical_cols = ['GENDER', 'RACE_ETHNICITY', 'FIRST_GEN_STATUS']

# Drop rows missing any of these key columns
df_train = df_training_assignment.dropna(subset=minmax_cols + standard_cols + categorical_cols).copy()
df_test = df_testing_assignment.dropna(subset=minmax_cols + standard_cols + categorical_cols).copy()

print(f"Training rows after dropping incomplete records: {len(df_train)}")

preprocessor = ColumnTransformer(
    transformers=[
        ('minmax',   MinMaxScaler(), minmax_cols),
        ('standard', StandardScaler(), standard_cols),
        ('onehot',   OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols),
    ],
    remainder='drop'
)

X_structured_train = preprocessor.fit_transform(df_train)
df_structured_train = pd.DataFrame(X_structured_train, index=df_train.index)

X_structured_test = preprocessor.transform(df_test)
df_structured_test = pd.DataFrame(X_structured_test, index=df_test.index)

print("Structured feature matrix:", df_structured_train.shape)



## 3. Reduce TF-IDF Text Features with PCA


Free-response text vectorized with TF-IDF produces hundreds of columns—far too many to cluster or model directly alongside our 19 structured features. If we don't reduce this dimension, the text data will mathematically "overwhelm" our numeric data (like GPA).

**PCA (Principal Component Analysis)** compresses the text variance into a small number of components. Our goal is to retain as much information as possible while significantly reducing the number of variables.

We begin by loading the TF-IDF matrix we generated in the previous module. This dataset contains the numerical representation of every important word across all student comments.



In [ ]:
ML_Survey_Data_Num = pd.read_csv(f'{filepath}ML_Survey_Data_Num.csv')
ML_Survey_Data22_Num = pd.read_csv(f'{filepath}ML_Survey_Data22_Num.csv')
display(ML_Survey_Data_Num)

In [ ]:
tfidf_matrix_train = ML_Survey_Data_Num.iloc[:,11:]
print("TF-IDF matrix:", tfidf_matrix_train.shape)
tfidf_matrix_train

In [ ]:
tfidf_matrix_test = ML_Survey_Data22_Num.iloc[:,11:]
print("TF-IDF matrix:", tfidf_matrix_test.shape)


**How many components do we actually need?** Instead of guessing, we use an **elbow plot** (Cumulative Explained Variance). In Institutional Research, a common threshold is **80%**. This means we want to find the minimum number of "Principal Components" that still capture 80% of the original meaning found in the hundreds of raw word columns.

In [ ]:
# Choose number of PCA components by explained variance
pca_full = PCA(random_state=42)
pca_full.fit(tfidf_matrix_train)

cumvar = np.cumsum(pca_full.explained_variance_ratio_)
threshold = 0.80
n_components = int(np.searchsorted(cumvar, threshold)) + 1

fig = px.line(
    x=range(1, len(cumvar) + 1), y=cumvar,
    labels={'x': 'Number of PCA Components', 'y': 'Cumulative Explained Variance'},
    title='PCA Explained Variance — TF-IDF Features'
)
fig.add_hline(y=threshold, line_dash='dash', annotation_text=f'{int(threshold*100)}% threshold')
fig.add_vline(x=n_components, line_dash='dot',
              annotation_text=f'{n_components} components', annotation_position='top right')
fig.show()
print(f"→ Using {n_components} PCA components to capture {threshold*100:.0f}% of text variance")

Now that we've identified that **33 components** are enough to represent our text data, we apply the final PCA transformation. We rename these new columns as `TEXT_PC1`, `TEXT_PC2`, etc., to signify that these are "compressed" features rather than raw word counts.

In [ ]:
# Apply PCA with the chosen number of components
pca = PCA(n_components=n_components, random_state=42)
X_text_pca_train = pca.fit_transform(tfidf_matrix_train)
df_text_pca_train = pd.DataFrame(X_text_pca_train, index=df_train.index,
                           columns=[f'TEXT_PC{i+1}' for i in range(n_components)])
print("Text PCA matrix:", df_text_pca_train.shape)


Having learned the number of components from the training data, we apply that transformation to the test data to find the same number of principal components *without learning from any test data* to avoid data leakage. These new variables will be needed in the modeling stage.

In [ ]:
# Apply PCA with the chosen number of components
X_text_pca_test = pca.transform(tfidf_matrix_test)
df_text_pca_test = pd.DataFrame(X_text_pca_test, index=df_test.index,
                           columns=[f'TEXT_PC{i+1}' for i in range(n_components)])
print("Text PCA matrix:", df_text_pca_test.shape)


## 4. Feature Fusion: Creating the Master Matrix


This is the "Final Assembly" step. We merge our scaled structured features (from Section 2) with our compressed text features (from Section 4) to create a unified student profile.

**The Result:** A single dataframe where every student is represented by a "digital fingerprint" that includes their demographics, their grades, and the core themes of their written feedback.

We use `pd.concat` with `axis=1` to join our two dataframes side-by-side.

> **Technical Note:** We cast the column names to strings at this stage. This ensures compatibility with the clustering and classification algorithms in the next course, which often require consistent header types to function correctly.

In [ ]:
df_all_train = pd.concat([df_structured_train, df_text_pca_train], axis=1)
df_all_train.columns = df_all_train.columns.astype(str)  # KMeans requires string column names
print("Combined feature matrix:", df_all_train.shape)
df_all_train

And for the test set:

In [ ]:
df_all_test = pd.concat([df_structured_test, df_text_pca_test], axis=1)
df_all_test.columns = df_all_test.columns.astype(str)  # KMeans requires string column names
print("Combined feature matrix:", df_all_test.shape)
df_all_test

When we used the `ColumnTransformer` earlier, it added prefixes like `minmax__` and `onehot__` to our column names. While helpful for tracking, these make for messy reports.

In this final step, we programmatically strip those prefixes to restore clean, readable headers (e.g., changing `minmax__HS_GPA` back to `HS_GPA`). This ensures our final Master Matrix is ready for both machine learning and human review.

In [ ]:
transformed_structured_feature_names = preprocessor.get_feature_names_out()

cleaned_structured_cols = []
for col_name in transformed_structured_feature_names:
    if col_name.startswith('minmax__'):
        cleaned_structured_cols.append(col_name.replace('minmax__', ''))
    elif col_name.startswith('standard__'):
        cleaned_structured_cols.append(col_name.replace('standard__', ''))
    elif col_name.startswith('onehot__'):
        # Remove the 'onehot__' prefix to make names like 'GENDER_Female' cleaner
        cleaned_structured_cols.append(col_name.replace('onehot__', ''))
    else:
        cleaned_structured_cols.append(col_name)

# The PCA column names are already correctly named in df_text_pca_train.columns
pca_cols = df_text_pca_train.columns.tolist()

# Combine all column names
df_all_train.columns = cleaned_structured_cols + pca_cols
df_all_test.columns = cleaned_structured_cols + pca_cols


print("Combined feature matrix with renamed columns:", df_all_train.shape)
display(df_all_train.head(3))

#print("Combined test feature matrix with renamed columns:", df_all_test.shape)
#display(df_all_test.head(3))

ML_Survey_Data_train = df_all_train
ML_Survey_Data_test = df_all_test

ML_Survey_Data_train['SEM_2_STATUS'] = df_training_assignment['SEM_2_STATUS']
ML_Survey_Data_test['SEM_2_STATUS'] = df_testing_assignment['SEM_2_STATUS']

ML_Survey_Data_train


Note that we did not include the entirety of the survey data. In the assignment, you'll be directed on how to use the **ordinal responses** to augment these DataFrames, and use them for downstream modeling.

In [ ]:
save_path = '/content/drive/MyDrive/IR ML Cert/MLCert Course 3/Course 3 Data/'

train = ML_Survey_Data_train
test = ML_Survey_Data_test

train.to_csv(save_path + 'ML_SURVEY_MASTER_TRAIN.csv', index=False)
test.to_csv(save_path + 'ML_SURVEY_MASTER_TEST.csv', index=False)


## 5. Wrap-Up




In this module, we achieved **three major** milestones:
1. **Multimodal Preprocessing:** Scaled and encoded numeric and categorical data simultaneously.
2. **Dimension Reduction:** Used PCA to distill 250+ text features into 33 high-impact components, preventing our text from "drowning out" our academic data.
3. **The Master Matrix:** Fused everything into a consolidated dataset that preserves both the student's metrics and their voice.

<br>

> **Instructor Perspective:** You are now ready to move from **Data Engineering** to **Advanced Modeling**. The `df_all` matrix you've built here is the exact input required for the predictive models and student segmentation strategies we will explore as we move forward.

<br>

